In [ ]:
from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# import pyspark as spark

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

# Notebooks live in EDA/, data lives in <repo root>/data/ (git-ignored -- never commit it).
DATA_DIR = Path('.') / 'data'



# Base DF + Enplanement

In [ ]:
base_df = pd.read_csv(DATA_DIR / 'made/AA_MASTER_V2.csv',index_col=0)
base_df.head()

In [ ]:


def create_grid_lags(
    df: pd.DataFrame,
    entity_cols: list,
    value_cols: list,
    lags: list = [12, 24],
    max_target_date: str = "2028-12-31",
    year_col: str = "Year",
    month_col: str = "Month",
    drop_raw: bool = True,
) -> pd.DataFrame:
    # Uses `df`, `year_col`, and `month_col` to build a proper timestamp
    work_df = df.copy()
    work_df["_date"] = pd.to_datetime(
        dict(year=work_df[year_col], month=work_df[month_col], day=1)
    )

    min_date = work_df["_date"].min()
    max_hist_date = work_df["_date"].max()
    # Uses `max_target_date` to set the end of the monthly calendar
    end_date = pd.to_datetime(max_target_date)

    # Uses `year_col` and `month_col` to build the complete monthly date grid
    all_dates = pd.date_range(start=min_date, end=end_date, freq="MS")
    date_grid = pd.DataFrame({
        year_col: all_dates.year,
        month_col: all_dates.month,
        "_date": all_dates,
    })

    # Initializes `grid` by cross-joining unique entities with every month in `date_grid`
    entities = work_df.loc[:, entity_cols].drop_duplicates()
    join_keys = list(entity_cols) + [year_col, month_col]
    
    grid = entities.merge(date_grid, how="cross").merge(
        work_df.drop(columns=["_date"]),
        on=join_keys,
        how="left",
    )

    # Fills missing historical months (<= max date in `df`) with 0 and keeps future rows (> T) as NaN
    hist_mask = grid["_date"] <= max_hist_date
    grid.loc[hist_mask, value_cols] = grid.loc[hist_mask, value_cols].fillna(0)

    # Sorts `grid` chronologically and shifts `value_cols` by each lag in `lags`
    sort_keys = list(entity_cols) + ["_date"]
    grid = grid.sort_values(sort_keys).reset_index(drop=True)

    for col in value_cols:
        for l in lags:
            grid[f"{col}_lag_{l}"] = grid.groupby(entity_cols)[col].shift(l)

    # Drops helper `_date` and optionally drops raw unlagged `value_cols`
    cols_to_drop = ["_date"] + (list(value_cols) if drop_raw else list())
    return grid.drop(columns=cols_to_drop)


In [ ]:
def left_join_feature_table(
    spine: pd.DataFrame,
    feature_df: pd.DataFrame,
    left_keys: list,
    right_keys: list = None,
    prefixes: list = None,
) -> pd.DataFrame:
    """
    Left-joins `feature_df` onto `spine` while strictly preserving row count.
    - For a single join : leave `prefixes=None`.
    - For a double join : pass
      `left_keys=[["orig_County_FIPS", "Macro_Year"], ["dest_County_FIPS", "Macro_Year"]]`
      and `prefixes=["orig_", "dest_"]`.
    """
    rows_before = len(spine)
    right_keys = right_keys or left_keys

    # Case A: Single Left-Join
    if prefixes is None:
        spine = spine.merge(
            feature_df,
            left_on=left_keys,
            right_on=right_keys,
            how="left",
        )
        extra_keys = [k for k in right_keys if k not in left_keys]
        if extra_keys:
            spine = spine.drop(columns=extra_keys)

    # Case B: Multi/Double Left-Join
    else:
        for l_keys, prefix in zip(left_keys, prefixes):
            prefixed_df = feature_df.add_prefix(prefix)
            prefixed_r_keys = [f"{prefix}{k}" for k in right_keys]

            spine = spine.merge(
                prefixed_df,
                left_on=l_keys,
                right_on=prefixed_r_keys,
                how="left",
            )
            # Only drop right-hand join keys whose names differed from `l_keys`
            extra_keys = [k for k in prefixed_r_keys if k not in l_keys]
            if extra_keys:
                spine = spine.drop(columns=extra_keys)

    assert len(spine) == rows_before, f"Row count exploded! {rows_before} -> {len(spine)}"
    return spine


In [ ]:

years = range(2017, 2026)
enplanement_dfs = []

for year in years:
  df = pd.read_csv(DATA_DIR / f'Enplanement{year}.csv')

  df.columns = df.columns.str.strip()

  # Find and match the column containing the current year and 'Landed Weight'
  enplan_col = [
      col
      for col in df.columns
      if str(year-2000) in col and f'cy {year-2000} enplanements' in col.lower()
  ][0]

  # Rename to standard column name and tag the year
  df = df.rename(columns={enplan_col: 'ENPLANEMENT'})
  df['YEAR'] = year

  df.drop(columns=[f'CY {year-2000} Enplanements', f'CY {year-2001} Enplanements', '% Change'], inplace=True, errors='ignore')
  enplanement_dfs.append(df)

# Combine all years into one master DataFrame
Enp= pd.concat(enplanement_dfs, ignore_index=True)
Enp.head()


In [ ]:
Enp.drop(columns = ['Rank','RO','ST','City','Unnamed: 11','Airport Name'],inplace = True)
Enp.head()

In [ ]:
Enp.rename(columns={'Locid':'LOCID','S/L':'ENP_SL','Hub':'ENP_HUB'},inplace=True)
Enp.head()


In [ ]:
Enp.dropna(subset=['ENPLANEMENT'],inplace=True)
len(Enp)

In [ ]:
base_df.rename(columns={'ORG_SVC_LVL':'ORG_CARGO_SL','DES_SVC_LVL':'DES_CARGO_SL','ORG_HUB':'ORG_CARGO_HUB',
                        'DES_HUB':'DES_CARGO_HUB'},inplace=True)
base_df.head()

In [ ]:

full_spine = base_df.copy()
# Create capped lookup year on your spine (same 2-year reporting lag rule as Cargo)
latest_enp_year = int(Enp["YEAR"].max())
full_spine["Enp_Lookup_Year"] = (full_spine["DEP_YEAR"] - 2).clip(upper=latest_enp_year)

# Double Left-Join onto Master Spine (Origin & Destination)
full_spine = left_join_feature_table(
    spine=full_spine,
    feature_df=Enp,
    left_keys=[["ORG_AIRPRT", "Enp_Lookup_Year"], ["DES_AIRPRT", "Enp_Lookup_Year"]],
    right_keys=["LOCID", "YEAR"],
    prefixes=["ORG_", "DES_"],
)

# Drop temporary helper column
full_spine = full_spine.drop(columns=["Enp_Lookup_Year"])


In [ ]:
full_spine.sample(5)

In [ ]:

full_spine.isnull().sum()/len(full_spine)

In [ ]:
full_spine.to_csv(DATA_DIR / 'made/AA_MASTER_V3.csv',index=False)

# 2.  PAX/REV SHIFT

In [ ]:
# Create 12-month and 24-month lags on your spine's target columns
full_spine = full_spine.sort_values(["ROUTE", "DEP_YEAR", "DEP_MONTH"]).reset_index(drop=True)

for col in ["OD_PAX_LOG", "OD_REV_USD_LOG"]:  # Update to match your exact column names
    full_spine[f"{col}_L24"] = full_spine.groupby("ROUTE")[col].shift(24)

# Add explicit COVID regime flags so 2020-2021 lags never mislead 2027-2028 forecasts
full_spine["L24_COVID"] = (full_spine["DEP_YEAR"] - 2).isin([2020, 2021]).astype(int)



In [ ]:
full_spine.sample(5)

In [ ]:
print(full_spine[full_spine.DEP_YEAR == 2022].head())
print(full_spine[full_spine.DEP_YEAR == 2020].head())

# 3. Airport info + Base DF

In [ ]:


# Load dim_airports with string dtypes so leading zeros in FIPS/CBSA codes stay intact
dim_airports = pd.read_csv(DATA_DIR / 
    "made/AIRPORT_V1.csv",  
    dtype={"County_FIPS": str, "CBSA_Code": str},
)

# Clean airport code whitespace & drop any duplicate airport rows
dim_airports["AIRPRT_CD"] = dim_airports["AIRPRT_CD"].astype(str).str.strip()
dim_airports = dim_airports.drop_duplicates(subset=["AIRPRT_CD"])

# Keep only the essential bridge & coordinate columns
keep_cols = dim_airports.columns
dim_airports = dim_airports[[c for c in keep_cols if c in dim_airports.columns]]

# Double Left-Join onto Master Spine 
full_spine = left_join_feature_table(
    spine=full_spine,
    feature_df=dim_airports,
    left_keys=[["ORG_AIRPRT"], ["DES_AIRPRT"]],
    right_keys=["AIRPRT_CD"],
    prefixes=["ORG_", "DES_"],
)

print("Null rates after dim_airports join:")
print(full_spine[["ORG_CNTRY_CD", "DES_CNTRY_CD", "ORG_CNTY_FIPS", "DES_CBSA_CD"]].isna().mean().round(3) * 100)


In [ ]:
full_spine.isnull().sum()/len(full_spine)

In [ ]:
base_df.isnull().value_counts()

In [ ]:
full_spine.to_csv(DATA_DIR / 'made/AA_MASTER_V4.csv',index=False)

# 4. County GDP / Income

In [ ]:
full_spine = pd.read_csv(DATA_DIR/'made/AA_MASTER_V4.csv')
full_spine.head()

In [ ]:

def clean_bea_wide_table(
    csv_path: str,
    line_code: int,
    value_col_name: str,
    min_year: int = 2015,
) -> pd.DataFrame:
    """
    Loads a BEA wide-format county table (CAGDP1 / CAINC1), filters to a single
    LineCode, cleans quoted GeoFIPS codes, drops state/national rollups ('*000'),
    melts year columns into long format, and converts BEA '(D)'/'(NA)' codes to NaN.
    """
    raw = pd.read_csv(csv_path, dtype={"GeoFIPS": str}, low_memory=False,encoding="cp1252")
    raw.columns = raw.columns.str.strip()

    # Clean GeoFIPS (strip quotes & whitespace) and drop national/state totals ending in 000
    raw["County_FIPS"] = (
        raw["GeoFIPS"]
        .astype(str)
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.zfill(5)
    )
    raw = raw[
        (raw["County_FIPS"].str.len() == 5)
        & (~raw["County_FIPS"].str.endswith("000"))
        & (pd.to_numeric(raw["LineCode"], errors="coerce") == line_code)
    ].copy()

    # 2. Identify all 4-digit year columns from min_year onward 
    year_cols = [c for c in raw.columns if c.isdigit() and int(c) >= min_year]

    # 3. Melt wide year columns into long ['County_FIPS', 'Year', value_col_name]
    long_df = raw.melt(
        id_vars=["County_FIPS"],
        value_vars=year_cols,
        var_name="Year",
        value_name=value_col_name,
    )

    long_df["Year"] = long_df["Year"].astype("Int64")
    long_df[value_col_name] = pd.to_numeric(long_df[value_col_name], errors="coerce")

    return long_df.drop_duplicates(subset=["County_FIPS", "Year"]).reset_index(drop=True)



In [ ]:


# 1. Clean & melt County Real GDP 
county_gdp_long = clean_bea_wide_table(
    csv_path=DATA_DIR / "County_GDP.csv",   
    line_code=1,
    value_col_name="CNTY_GDP",
    min_year=2015,
)

# 2. Clean & melt County Per Capita Income 
county_inc_long = clean_bea_wide_table(
    csv_path=DATA_DIR / "County_Income.csv", 
    line_code=3,
    value_col_name="CNTY_INC",
    min_year=2015,
)

# 3. Combine GDP & Income into a single lookup table
county_macro = county_gdp_long.merge(
    county_inc_long,
    on=["County_FIPS", "Year"],
    how="outer",
)

# 4. Standardize FIPS codes on your Master Spine (5-digit zero-padded string)
for fips_col in ["ORG_CNTY_FIPS", "DES_CNTY_FIPS"]:
    if fips_col in full_spine.columns:
        full_spine[fips_col] = full_spine[fips_col].astype(str).str.strip()
        valid_mask = ~full_spine[fips_col].isin(["nan", "None", "MISSING", ""])
        full_spine.loc[valid_mask, fips_col] = full_spine.loc[valid_mask, fips_col].str.zfill(5)

# 5. Create capped Macro_Year (since BEA stops at 2024, (DEP_YEAR - 2).clip(upper=2024)
latest_bea_year = int(county_macro["Year"].max()) 
print("Latest BEA County Year:", latest_bea_year)

full_spine["Macro_Year"] = (full_spine["DEP_YEAR"] - 2).clip(upper=latest_bea_year)



In [ ]:
county_macro.CNTY_GDP *= 1000
county_macro.CNTY_INC
county_macro.head()

In [ ]:
county_macro.CNTY_GDP = np.log1p(county_macro.CNTY_GDP)
county_macro.head()

In [ ]:
county_macro.rename(columns={'CNTY_GDP':'CNTY_GDP_LOG'},inplace=True)
county_macro.head()

In [ ]:
# Double Left-Join onto Master Spine (Origin & Destination)


full_spine = left_join_feature_table(
    spine=full_spine,
    feature_df=county_macro,
    left_keys=[['ORG_CNTY_FIPS', "Macro_Year"], ["DES_CNTY_FIPS", "Macro_Year"]],
    right_keys=["County_FIPS", "Year"],
    prefixes=["ORG_", "DES_"],
)

full_spine = full_spine.drop(columns=["Macro_Year"])
full_spine.head()


In [ ]:
full_spine.isnull().sum()/len(full_spine)

In [ ]:
# Get 1 row per unique route
unique_routes = full_spine[["ROUTE", "ORG_CNTRY_CD", "DES_CNTRY_CD"]].drop_duplicates(subset=["ROUTE"])

# Check if EITHER Origin OR Destination is in the US
us_mask = (unique_routes["ORG_CNTRY_CD"] == "US") | (unique_routes["DES_CNTRY_CD"] == "US")

total_routes = len(unique_routes)
us_touching = us_mask.sum()

print(f"Total unique routes: {total_routes:,}")
print(f"Routes with at least 1 US stop: {us_touching:,} ({us_touching / total_routes * 100:.1f}%)")
print(f"Foreign-to-Foreign routes (0 US stops): {total_routes - us_touching:,} ({(1 - us_touching / total_routes) * 100:.1f}%)")


In [ ]:
full_spine.ORG_CNTY_INC.isnull().value_counts()

In [ ]:
county_gdp_long.isnull().value_counts()

In [ ]:
county_inc_long.isnull().value_counts()

In [ ]:
full_spine.to_csv(DATA_DIR/'made/AA_MASTER_V5.csv',index=False)

# 5. Country GDP Per Capita + Base df

In [ ]:
import pandas as pd

# 1. Comprehensive ISO-2 -> ISO-3 lookup dictionary (covers all major aviation countries & territories)
ISO2_TO_ISO3 = {
    "US": "USA", "PR": "PRI", "GU": "GUM", "VI": "VIR", "AS": "ASM", "MP": "MNP",
    "KR": "KOR", "JP": "JPN", "CN": "CHN", "TW": "TWN", "HK": "HKG", "MO": "MAC",
    "SG": "SGP", "TH": "THA", "VN": "VNM", "PH": "PHL", "MY": "MYS", "ID": "IDN",
    "IN": "IND", "AU": "AUS", "NZ": "NZL", "CA": "CAN", "MX": "MEX", "BR": "BRA",
    "AR": "ARG", "CL": "CHL", "CO": "COL", "PE": "PER", "PA": "PAN", "CR": "CRI",
    "DO": "DOM", "JM": "JAM", "BS": "BHS", "AW": "ABW", "CW": "CUW", "KY": "CYM",
    "GB": "GBR", "FR": "FRA", "DE": "DEU", "NL": "NLD", "ES": "ESP", "IT": "ITA",
    "CH": "CHE", "AT": "AUT", "BE": "BEL", "IE": "IRL", "PT": "PRT", "SE": "SWE",
    "NO": "NOR", "DK": "DNK", "FI": "FIN", "PL": "POL", "CZ": "CZE", "GR": "GRC",
    "TR": "TUR", "RU": "RUS", "UA": "UKR", "AE": "ARE", "QA": "QAT", "SA": "SAU",
    "IL": "ISR", "EG": "EGY", "ZA": "ZAF", "ET": "ETH", "KE": "KEN", "NG": "NGA",
    "MN": "MNG", "KH": "KHM", "LA": "LAO", "MM": "MMR", "UZ": "UZB", "KZ": "KAZ",
}


def map_iso2_to_iso3(series: pd.Series) -> pd.Series:
    """Maps 2-letter ISO country codes to 3-letter ISO codes (leaves existing 3-letter codes intact)."""
    cleaned = series.astype(str).str.strip().str.upper()
    mapped = cleaned.map(ISO2_TO_ISO3)
    # If already 3 letters (e.g., 'USA'), keep as-is; otherwise use mapped value
    return cleaned.where(cleaned.str.len() == 3, mapped)


# 2. Apply to spine's Origin and Destination country codes
for col in ["ORG_CNTRY_CD", "DES_CNTRY_CD"]:
    full_spine[col] = map_iso2_to_iso3(full_spine[col])

unmapped = (
    set(full_spine["ORG_CNTRY_CD"].dropna()) | set(full_spine["DES_CNTRY_CD"].dropna())
) - set(ISO2_TO_ISO3.values()) - {"NAN", "NONE", ""}
print("Unmapped country codes remaining:", unmapped)


In [ ]:
full_spine.head()

In [ ]:

# 1. Load Country GDP per Capita CSV 
country_raw = pd.read_csv(DATA_DIR/"country_gdp_per_capita.csv")
country_raw.columns = country_raw.columns.str.strip()


country_long = country_raw.rename(columns={
    "year":"DEP_YEAR",
    "iso3_country": "CNTRY_CD",  
    "gdp_per_capita": "CNTRY_GDP",
})[["CNTRY_CD", "DEP_YEAR", "CNTRY_GDP"]].copy()

# 2. Clean keys, convert to numeric, and apply consistent log1p scaling
country_long["CNTRY_CD"] = country_long["CNTRY_CD"].astype(str).str.strip().str.upper()
country_long["DEP_YEAR"] = pd.to_numeric(country_long["DEP_YEAR"], errors="coerce").astype("Int64")
country_long["CNTRY_GDP"] = pd.to_numeric(country_long["CNTRY_GDP"], errors="coerce")

country_long = (
    country_long.dropna(subset=["CNTRY_CD", "DEP_YEAR", "CNTRY_GDP"])
    .drop_duplicates(subset=["CNTRY_CD", "DEP_YEAR"])
)
country_long["CNTRY_GDP"] = np.log1p(country_long["CNTRY_GDP"].clip(lower=0))

# 3. Create capped Macro_Year (anchors 2027-2028 on your latest published year)
latest_country_year = int(country_long["DEP_YEAR"].max())
print("Latest Country GDP Year:", latest_country_year)

full_spine["Macro_Year"] = (full_spine["DEP_YEAR"] - 2).clip(upper=latest_country_year)

# Standardize spine country codes before merging
for col in ["ORG_CNTRY_CD", "DES_CNTRY_CD"]:
    full_spine[col] = full_spine[col].astype(str).str.strip().str.upper()

# 4. Double Left-Join onto Master Spine 
full_spine1 = left_join_feature_table(
    spine=full_spine,
    feature_df=country_long,
    left_keys=[["ORG_CNTRY_CD", "Macro_Year"], ["DES_CNTRY_CD", "Macro_Year"]],
    right_keys=["CNTRY_CD", "DEP_YEAR"],
    prefixes=["ORG_", "DES_"],
).drop(columns=["Macro_Year"])

print("Country GDP null %:", full_spine1[["ORG_CNTRY_GDP", "DES_CNTRY_GDP"]].isna().mean().round(3) * 100)


In [ ]:
full_spine1.isnull().sum()/len(full_spine1)

In [ ]:
full_spine = full_spine1.copy()

In [ ]:
full_spine.to_csv(DATA_DIR/'made/AA_MASTER_V6.csv',index=False)

# 6. US/Puerto Rico CBSA

In [ ]:
full_spine = pd.read_csv(DATA_DIR/'made/AA_MASTER_V6.csv')

In [ ]:
# When MDIV is present, use MDIV as the 5-digit lookup code

cbsa_pr = pd.read_csv(DATA_DIR/"prc-cbsa-est2025.csv", dtype={"CBSA": str, "STCOU": str}, encoding="cp1252")
cbsa_us = pd.read_csv(DATA_DIR/"cbsa-est2025-alldata.csv", dtype={"CBSA": str, "STCOU": str, "MDIV": str}, encoding="cp1252")

cbsa_raw = pd.concat([cbsa_us, cbsa_pr], ignore_index=True)
cbsa_raw.columns = cbsa_raw.columns.str.strip()

# Drop county sub-rows where STCOU is not null but keep Metro Statistical Areas, Micro Areas, Metro Divisions
cbsa_raw = cbsa_raw[cbsa_raw["STCOU"].isna()].copy()

if "MDIV" in cbsa_raw.columns:
    cbsa_raw["Lookup_Code"] = cbsa_raw["MDIV"].fillna(cbsa_raw["CBSA"])
else:
    cbsa_raw["Lookup_Code"] = cbsa_raw["CBSA"]

cbsa_raw["CBSA_CD"] = (
    cbsa_raw["Lookup_Code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)  
    .str.strip()
    .str.zfill(5)
)
cbsa_raw = cbsa_raw.dropna(subset=["CBSA_CD"]).drop_duplicates(subset=["CBSA_CD"])

pop_cols = sorted([c for c in cbsa_raw.columns if c.startswith("POPESTIMATE") and c[-4:].isdigit()])
cbsa_long = cbsa_raw.melt(
    id_vars=["CBSA_CD"],
    value_vars=pop_cols,
    var_name="Year_Str",
    value_name="CBSA_Pop_Raw",
)
cbsa_long["Year"] = cbsa_long["Year_Str"].str.replace("POPESTIMATE", "", regex=False).astype("Int64")
cbsa_long["CBSA_Pop_Raw"] = pd.to_numeric(cbsa_long["CBSA_Pop_Raw"], errors="coerce")

netmig_cols = sorted([c for c in cbsa_raw.columns if c.startswith("NETMIG") and c[-4:].isdigit()])
if netmig_cols:
    mig_long = cbsa_raw.melt(
        id_vars=["CBSA_CD"],
        value_vars=netmig_cols,
        var_name="Year_Str",
        value_name="CBSA_Net_Mig",
    )
    mig_long["Year"] = mig_long["Year_Str"].str.replace("NETMIG", "", regex=False).astype("Int64")
    mig_long["CBSA_Net_Mig"] = pd.to_numeric(mig_long["CBSA_Net_Mig"], errors="coerce")
    cbsa_long = cbsa_long.merge(mig_long[["CBSA_CD", "Year", "CBSA_Net_Mig"]], on=["CBSA_CD", "Year"], how="left")
    cbsa_long["CBSA_Net_Mig_Rate"] = cbsa_long["CBSA_Net_Mig"] / cbsa_long["CBSA_Pop_Raw"].replace(0, pd.NA)
else:
    cbsa_long["CBSA_Net_Mig_Rate"] = np.nan

cbsa_long = cbsa_long.sort_values(["CBSA_CD", "Year"]).reset_index(drop=True)
cbsa_long["CBSA_Pop_Growth_1yr"] = cbsa_long.groupby("CBSA_CD")["CBSA_Pop_Raw"].pct_change(1)

# Backfill 2020's missing 1 year growth rate with 2021's growth rate so 2019-2022 rows aren't NaN
cbsa_long["CBSA_Pop_Growth_1yr"] = cbsa_long.groupby("CBSA_CD")["CBSA_Pop_Growth_1yr"].bfill()
cbsa_long["CBSA_Pop"] = np.log1p(cbsa_long["CBSA_Pop_Raw"].clip(lower=0))

cbsa_clean = cbsa_long[["CBSA_CD", "Year", "CBSA_Pop", "CBSA_Pop_Growth_1yr", "CBSA_Net_Mig_Rate"]].dropna(subset=["CBSA_Pop"])

# Drop the old partial CBSA columns from full_spine before re-joining
old_cbsa_cols = [c for c in full_spine.columns if "CBSA_Pop" in c or "CBSA_Net_Mig" in c]
full_spine = full_spine.drop(columns=old_cbsa_cols, errors="ignore")

# Standardize spine CBSA codes (strip '.0' if float!) & re-join
for col in ["ORG_CBSA_CD", "DES_CBSA_CD"]:
    full_spine[col] = full_spine[col].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    valid_mask = ~full_spine[col].isin(["nan", "None", "MISSING", "", "<NA>"])
    full_spine.loc[valid_mask, col] = full_spine.loc[valid_mask, col].str.zfill(5)

full_spine["CBSA_Lookup_Year"] = (full_spine["DEP_YEAR"] - 2).clip(lower=2020, upper=2025)


In [ ]:

full_spine = left_join_feature_table(
    spine=full_spine,
    feature_df=cbsa_clean,
    left_keys=[["ORG_CBSA_CD", "CBSA_Lookup_Year"], ["DES_CBSA_CD", "CBSA_Lookup_Year"]],
    right_keys=["CBSA_CD", "Year"],
    prefixes=["ORG_", "DES_"],
).drop(columns=["CBSA_Lookup_Year"])

print("Updated CBSA Pop null %:", full_spine[["ORG_CBSA_Pop", "DES_CBSA_Pop"]].isna().mean().round(3) * 100)


In [ ]:
full_spine.rename(columns={'ORG_CBSA_Net_Mig_Rate':'ORG_CBSA_NET_MIG','ORG_CBSA_Pop_Growth_1yr':'ORG_CBSA_POP_GRWT',
                           'ORG_CBSA_Pop': 'ORG_CBSA_POP','DES_CBSA_Net_Mig_Rate':'DES_CBSA_NET_MIG',
                           'DES_CBSA_Pop_Growth_1yr':'DES_CBSA_POP_GRWT',
                           'DES_CBSA_Pop': 'DES_CBSA_POP'},inplace=True)
full_spine.isnull().sum()/len(full_spine)

In [ ]:
full_spine.to_csv(DATA_DIR/'made/AA_MASTER_V7.csv',index=False)


# 7. County to county migrations

In [ ]:
full_spine = pd.read_csv(DATA_DIR/'made/AA_MASTER_V7.csv')

In [ ]:

# 1. Load the Net In/Out CSV
mig_raw = pd.read_csv(
    DATA_DIR/"CTC2014-2018_inout_net.csv",  # Update to your exact 'in and out' filename
    skiprows=3,
    header=None,
    dtype=str,
    encoding="cp1252",
)

# 2. Extract Geography A FIPS, Geography B FIPS, and the 4 Estimate columns
mig = pd.DataFrame({
    "state_a": mig_raw[0].str.strip(),
    "county_a": mig_raw[1].str.strip(),
    "state_b": mig_raw[2].str.strip(),
    "county_b": mig_raw[3].str.strip(),
    "flow_b_to_a": pd.to_numeric(mig_raw[8], errors="coerce"),
    "flow_a_to_b": pd.to_numeric(mig_raw[10], errors="coerce"),
    "net_b_to_a": pd.to_numeric(mig_raw[12], errors="coerce"),
    "gross_ab": pd.to_numeric(mig_raw[14], errors="coerce"),
})

# Keep only valid US/PR county-to-county rows (drops footer notes and non-county foreign region rows)
mig = mig[
    mig["state_a"].str.isdigit()
    & mig["county_a"].str.isdigit()
    & mig["state_b"].str.isdigit()
    & mig["county_b"].str.isdigit()
].copy()

# Census 2014-2018 uses 3-digit state codes ('001') + 3-digit county codes ('003')
# take last 2 digits of state + 3 digits of county = 5-digit FIPS ('01003')
mig["fips_a"] = mig["state_a"].str[-2:].str.zfill(2) + mig["county_a"].str.zfill(3)
mig["fips_b"] = mig["state_b"].str[-2:].str.zfill(2) + mig["county_b"].str.zfill(3)

# 3. Build symmetric lookup so flight directions A -> B and B -> A get matched
dir_ab = pd.DataFrame({
    "orig_County_FIPS": mig["fips_a"],
    "dest_County_FIPS": mig["fips_b"],
    "Mig_Outbound_Flow": mig["flow_a_to_b"],  # Movers from Orig -> Dest
    "Mig_Inbound_Flow": mig["flow_b_to_a"],   # Movers from Dest -> Orig
    "Mig_Net_Into_Orig": mig["net_b_to_a"],   # Net flow into Orig from Dest
    "Mig_Gross_Flow": mig["gross_ab"],        # Total two-way movers
})

dir_ba = pd.DataFrame({
    "orig_County_FIPS": mig["fips_b"],
    "dest_County_FIPS": mig["fips_a"],
    "Mig_Outbound_Flow": mig["flow_b_to_a"],  # Movers from Orig (B) -> Dest (A)
    "Mig_Inbound_Flow": mig["flow_a_to_b"],   # Movers from Dest (A) -> Orig (B)
    "Mig_Net_Into_Orig": -mig["net_b_to_a"],  # Reversed sign for Net flow into B
    "Mig_Gross_Flow": mig["gross_ab"],
})

mig_symmetric = (
    pd.concat([dir_ab, dir_ba], ignore_index=True)
    .groupby(["orig_County_FIPS", "dest_County_FIPS"], as_index=False)
    .mean()
)

# 4. Apply consistent log1p & signed-log transformations
mig_symmetric["Mig_Outbound_log1p"] = np.log1p(mig_symmetric["Mig_Outbound_Flow"].clip(lower=0))
mig_symmetric["Mig_Inbound_log1p"] = np.log1p(mig_symmetric["Mig_Inbound_Flow"].clip(lower=0))
mig_symmetric["Mig_Gross_log1p"] = np.log1p(mig_symmetric["Mig_Gross_Flow"].clip(lower=0))
mig_symmetric["Mig_Net_signed_log"] = (
    np.sign(mig_symmetric["Mig_Net_Into_Orig"]) * np.log1p(mig_symmetric["Mig_Net_Into_Orig"].abs())
)

mig_final = mig_symmetric[[
    "orig_County_FIPS", "dest_County_FIPS",
    "Mig_Outbound_log1p", "Mig_Inbound_log1p", "Mig_Gross_log1p", "Mig_Net_signed_log",
]]



In [ ]:
mig_final.rename(columns={'orig_County_FIPS':'ORG_CNTY_FIPS','dest_County_FIPS':'DES_CNTY_FIPS',
                          'Mig_Outbound_log1p':'MIG_OUT_LOG','Mig_Inbound_log1p':'MIG_IN_LOG',
                          'Mig_Gross_log1p':'MIG_GROSS_LOG','Mig_Net_signed_log':'MIG_NET_LOG'},inplace=True)
mig_final.head()

In [ ]:

# Left-join once onto Master Spine
full_spine = left_join_feature_table(
    spine=full_spine,
    feature_df=mig_final,
    left_keys=["ORG_CNTY_FIPS", "DES_CNTY_FIPS"],
)


In [ ]:
full_spine.columns

In [ ]:

# Fill 0.0 on US-to-US domestic county pairs with zero Census sample movers ad keep international routes as NaN
us_domestic = (full_spine["ORG_CNTRY_CD"] == "USA") & (full_spine["DES_CNTRY_CD"] == "USA")
for col in ["MIG_OUT_LOG", "MIG_IN_LOG", "MIG_GROSS_LOG", "MIG_NET_LOG"]:
    full_spine.loc[us_domestic, col] = full_spine.loc[us_domestic, col].fillna(0.0)

print("US-to-US Migration non-null %:", (full_spine.loc[us_domestic, "MIG_GROSS_LOG"].notna().mean() * 100).round(1))

In [ ]:
full_spine.to_csv(DATA_DIR/'made/AA_MASTER_V8.csv',index=False)

In [ ]:
full_spine.shape

In [ ]:
full_spine.isna().mean()

In [ ]:
full_spine.columns

# 8. Final Clean

In [ ]:
full_spine.drop(columns=['ORG_CARGO_SL','ORG_CARGO_HUB','DES_CARGO_SL','DES_CARGO_HUB','ORG_CNTRY_CD',
'ORG_CNTY_FIPS','ORG_CBSA_CD','DES_CNTRY_CD','DES_CNTY_FIPS','DES_CBSA_CD'],inplace=True)
full_spine.rename(columns={'ORG_ENP_HUB':'ORG_HUB','ORG_ENP_SL':'ORG_SL','DES_ENP_HUB':'DES_HUB','DES_ENP_SL':'DES_SL'},inplace=True)
full_spine.columns

In [ ]:
full_spine.isna().mean()

In [ ]:
full_spine.to_csv(DATA_DIR/'made/AA_MASTER_V9.csv',index=False)